# Notebook de Generación del Dataset de Entrenamiento

## 1. Selección del Dataset

Los datasets base seleccionados para la confección del dataset final para el entrenamiento de la IA de detección SMishing de Posdata son los siguientes:
- Kaggle SMS Spam Dataset (en inglés)
- Hugging Face softecapps's SMS Spam Dataset (en español, compuesto por un conjunto de entrenamiento y otro de test, aunque los usaremos como uno solo)

## 2. Traducción de los datasets

Puesto que contamos con mensajes tanto en inglés como en español, realizaremos un detectos que aprenda de ambos idiomas, de modo que el dataset en inglés será traducido también al español y, consiguientemente, el dataset en español será traducido al inglés.


### 2.1. Traducción al español del Dataset de Kaggle

In [ ]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
from google.colab import drive
import os

# Constants
DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
ENGLISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_kaggle.csv')
SPANISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_raw.csv')
BATCH_SIZE = 100
MODEL_NAME = 'Helsinki-NLP/opus-mt-en-es'

# Auxiliary functions
def parse_broken_row(line):
  '''
  Parses a broken CSV row into label and text components.

  Parameters:
  ----------
  line : str
      A single line from the CSV file.
  
  Returns:
  -------
  tuple or None
      A tuple (label, text) if parsing is successful, otherwise None.
  '''
  line = line.strip()

  while line.endswith(';') or line.endswith(','):
    line = line[:-1]
  
  if line.startswith('"') and line.endswith('"'):
    line = line[1:-1]

  parts = line.split(',', 1)
  if len(parts) < 2:
    return None
  
  label = parts[0].strip()
  text = parts[1].strip()

  text = text.strip(' ,;')
  if text.startswith('""') and text.endswith('""'):
    text = text[2:-2].replace('""', '"')
  elif text.startswith('"') and text.endswith('"'):
    text = text[1:-1].replace('""', '"')  
  
  return label, text

def translate_batch(texts, tokenizer, model, device):
    '''
    Translates a batch of texts from English to Spanish.

    Parameters:
    ----------
    texts : list of str
        List of English texts to translate.
    tokenizer : MarianTokenizer
        The tokenizer for the translation model.
    model : MarianMTModel
        The translation model.
    device : str
        The device to run the model on ('cpu' or 'cuda').

    Returns:
    -------
    list of str
        The translated texts.
    '''
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]


# Execution starts HERE

# Mount Google Drive
drive.mount('/content/drive')

data = [] # List to hold parsed data
try:
  with open(ENGLISH_FILE, 'r', encoding='latin-1') as file: # Open the dataset file
    for line in file: # Read each line
      parsed_row = parse_broken_row(line) # Parse the line
      if parsed_row: # If parsing was successful
        label, text = parsed_row # Unpack the tuple
        if label in ['ham', 'spam']: # Validate label
          data.append({'label': label, 'text': text}) # Append to data list
    dataset = pd.DataFrame(data) # Create DataFrame from data list
    print(f"Dataset loaded successfully. Total rows: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

# Load translation model and tokenizer
print(f"Loading model {MODEL_NAME}...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

print("Starting translation...")

spanish_texts = [] # List to hold translated texts
total_texts = len(dataset) # Total number of texts to translate

for i in tqdm(range(0, total_texts, BATCH_SIZE)): # Process texts in batches
    batch_texts = dataset['text'].iloc[i:i + BATCH_SIZE].tolist() # Get batch of texts
    translated_texts = translate_batch(batch_texts, tokenizer, model, device) # Translate batch
    spanish_texts.extend(translated_texts) # Append translated texts to list

dataset['text_es'] = spanish_texts # Add translated texts to DataFrame

# Display some example translations
print("Example translations:")
print(dataset[['text', 'text_es']].head())

dataset_final = dataset[['label', 'text_es']].rename(columns={'text_es': 'text'}) # Prepare final dataset
dataset_final.to_csv(SPANISH_FILE, index=False) # Save to CSV

print(f"Translation completed. Translated dataset saved to {SPANISH_FILE}.")

In [ ]:
import html

# Constants
SPANISH_FILE_RAW = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_raw.csv')
SPANISH_FILE_CLEAN = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_clean.csv')

# Auxiliary function
def clean_text(text):
    '''
    Cleans text by unescaping HTML entities and fixing encoding issues.

    Parameters:
    ----------
    text : str
        The text to clean.

    Returns:
    -------
    str
        The cleaned text.
    '''
    if not isinstance(text, str):
        return str(text)
    
    text = html.unescape(text)
    
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
        
    return text

# Execution starts HERE

print(f"Loading {SPANISH_FILE_RAW}...")
df = pd.read_csv(SPANISH_FILE_RAW) # Load the raw Spanish dataset

print("Cleaning text...")
df['text'] = df['text'].apply(clean_text) # Clean the text column

df.to_csv(SPANISH_FILE_CLEAN, index=False, encoding='utf-8-sig') # Save cleaned dataset

print(f"Done! Cleaned dataset saved as: {SPANISH_FILE_CLEAN}")

### 2.2. Traducción al inglés del Dataset de Hugging Face

In [ ]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
from google.colab import drive
import os

# Constants
DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
SPANISH_FILE_1 = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_huggingface_1.csv')
ENGLISH_FILE_1 = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_huggingface_1.csv')
SPANISH_FILE_2 = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_huggingface_2.csv')
ENGLISH_FILE_2 = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_huggingface_2.csv')
BATCH_SIZE = 100
MODEL_NAME = 'Helsinki-NLP/opus-mt-es-en'

# Auxiliary functions
def translate_batch(texts, tokenizer, model, device):
    '''
    Translates a batch of texts from Spanish to English.

    Parameters:
    ----------
    texts : list of str
        List of Spanish texts to translate.
    tokenizer : MarianTokenizer
        The tokenizer for the translation model.
    model : MarianMTModel
        The translation model.
    device : str
        The device to run the model on ('cpu' or 'cuda').

    Returns:
    -------
    list of str
        The translated texts.
    '''
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

# Execution starts HERE

# Mount Google Drive
drive.mount('/content/drive')

try:
  dataset1 = pd.read_csv(SPANISH_FILE_1) # Open the first dataset file
  dataset2 = pd.read_csv(SPANISH_FILE_2) # Open the second dataset file
  dataset1.rename(columns={'mensaje': 'text', 'tipo': 'label'}, inplace=True) # Rename columns of first dataset
  dataset2.rename(columns={'mensaje': 'text', 'tipo': 'label'}, inplace=True) # Rename columns of second dataset
  print(f"Dataset loaded successfully. Total rows: {len(dataset1) + len(dataset2)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

# Load translation model and tokenizer
print(f"Loading model {MODEL_NAME}...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

print("Starting translation...")

english_texts_1 = [] # List to hold translated texts of first dataset
english_texts_2 = [] # List to hold translated texts of second dataset
total_texts_1 = len(dataset1) # Total number of texts to translate in first dataset
total_texts_2 = len(dataset2) # Total number of texts to translate in second dataset

for i in tqdm(range(0, total_texts_1, BATCH_SIZE)): # Process texts in batches
    batch_texts = dataset1['text'].iloc[i:i + BATCH_SIZE].tolist() # Get batch of texts
    translated_texts = translate_batch(batch_texts, tokenizer, model, device) # Translate batch
    english_texts_1.extend(translated_texts) # Append translated texts to list

dataset1['text_en'] = english_texts_1 # Add translated texts to DataFrame
# Display some example translations
print("Example translations in dataset 1:")
print(dataset1[['text', 'text_en']].head())
dataset_final_1 = dataset1[['label', 'text_en']].rename(columns={'text_en': 'text'}) # Prepare final dataset
dataset_final_1.to_csv(ENGLISH_FILE_1, index=False) # Save to CSV

for i in tqdm(range(0, total_texts_2, BATCH_SIZE)): # Process texts in batches
    batch_texts = dataset2['text'].iloc[i:i + BATCH_SIZE].tolist() # Get batch of texts
    translated_texts = translate_batch(batch_texts, tokenizer, model, device) # Translate batch
    english_texts_2.extend(translated_texts) # Append translated texts to list

dataset2['text_en'] = english_texts_2 # Add translated texts to DataFrame
# Display some example translations
print("Example translations in dataset 2:")
print(dataset2[['text', 'text_en']].head())
dataset_final_2 = dataset2[['label', 'text_en']].rename(columns={'text_en': 'text'}) # Prepare final dataset
dataset_final_2.to_csv(ENGLISH_FILE_2, index=False) # Save to CSV

print(f"Translation completed. Translated dataset saved to {ENGLISH_FILE_1} and {ENGLISH_FILE_2}.")

### 2.3. Arreglo y limpieza de los datasets originales

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Constants
DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
ENGLISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_kaggle.csv')

# Auxiliary functions
def parse_broken_row(line):
  '''
  Parses a broken CSV row into label and text components.

  Parameters:
  ----------
  line : str
      A single line from the CSV file.
  
  Returns:
  -------
  tuple or None
      A tuple (label, text) if parsing is successful, otherwise None.
  '''
  line = line.strip()

  while line.endswith(';') or line.endswith(','):
    line = line[:-1]
  
  if line.startswith('"') and line.endswith('"'):
    line = line[1:-1]

  parts = line.split(',', 1)
  if len(parts) < 2:
    return None
  
  label = parts[0].strip()
  text = parts[1].strip()

  text = text.strip(' ,;')
  if text.startswith('""') and text.endswith('""'):
    text = text[2:-2].replace('""', '"')
  elif text.startswith('"') and text.endswith('"'):
    text = text[1:-1].replace('""', '"')  
  
  return label, text

# Mount Google Drive
drive.mount('/content/drive')

data = [] # List to hold parsed data
dataset = pd.DataFrame() # Empty DataFrame
try:
  with open(ENGLISH_FILE, 'r', encoding='latin-1') as file: # Open the dataset file
    for line in file: # Read each line
      parsed_row = parse_broken_row(line) # Parse the line
      if parsed_row: # If parsing was successful
        label, text = parsed_row # Unpack the tuple
        if label in ['ham', 'spam']: # Validate label
          data.append({'label': label, 'text': text}) # Append to data list
    dataset = pd.DataFrame(data) # Create DataFrame from data list
    print(f"Dataset loaded successfully. Total rows: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

dataset.to_csv(ENGLISH_FILE, index=False) # Save cleaned dataset

print(f"Done! Cleaned dataset saved as: {ENGLISH_FILE}")

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Constants
DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
SPANISH_FILE_1 = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_huggingface_1.csv')
SPANISH_FILE_2 = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_huggingface_2.csv')

#Mount Google Drive
drive.mount('/content/drive')

try:
  dataset1 = pd.read_csv(SPANISH_FILE_1) # Open the first dataset file
  dataset2 = pd.read_csv(SPANISH_FILE_2) # Open the second dataset file
  dataset1.rename(columns={'mensaje': 'text', 'tipo': 'label'}, inplace=True) # Rename columns of first dataset
  dataset2.rename(columns={'mensaje': 'text', 'tipo': 'label'}, inplace=True) # Rename columns of second dataset
  print(f"Dataset loaded successfully. Total rows: {len(dataset1) + len(dataset2)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

dataset1.to_csv(SPANISH_FILE_1, index=False) # Save cleaned dataset 1
dataset2.to_csv(SPANISH_FILE_2, index=False) # Save cleaned dataset 2

print(f"Done! Cleaned datasets saved as: {SPANISH_FILE_1} and {SPANISH_FILE_2}")

### 3.4. Combinación de todos los datasets en uno solo unificado

In [ ]:
import pandas as pd
import os
from google.colab import drive

# Constants
DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
DIR_EN = os.path.join(DRIVE_PATH, 'datasets', 'en')
DIR_ES = os.path.join(DRIVE_PATH, 'datasets', 'es')
OUTPUT_FILE = os.path.join(DRIVE_PATH, 'datasets', 'bilingual_spam_sms_dataset.csv')

drive.mount('/content/drive') # Mount Google Drive

# Define dataset file pairs
pairs = [
    {
        'en': os.path.join(DIR_EN, 'spam_sms_english_kaggle.csv'),
        'es': os.path.join(DIR_ES, 'spam_sms_spanish_kaggle_clean.csv'),
        'source': 'kaggle'
    },
    {
        'en': os.path.join(DIR_EN, 'spam_sms_english_huggingface_1.csv'),
        'es': os.path.join(DIR_ES, 'spam_sms_spanish_huggingface_1.csv'),
        'source': 'huggingface_1'
    },
    {
        'en': os.path.join(DIR_EN, 'spam_sms_english_huggingface_2.csv'),
        'es': os.path.join(DIR_ES, 'spam_sms_spanish_huggingface_2.csv'),
        'source': 'huggingface_2'
    }
]

dfs_to_merge = []

print("Starting to merge datasets...")

for p in pairs:
    try:
        print(f"Loading pair from source: {p['source']}")
        df_en = pd.read_csv(p['en'])
        df_es = pd.read_csv(p['es'])

        df_en.columns = df_en.columns.str.strip().str.lower()
        df_es.columns = df_es.columns.str.strip().str.lower()

        if len(df_en) != len(df_es):
            print(f"Warning: Mismatched row counts in {p['source']} (EN: {len(df_en)}, ES: {len(df_es)})")
            exit(1)
        
        merged_df = pd.DataFrame({
            'label': df_en['label'],
            'text_en': df_en['text'],
            'text_es': df_es['text'],
            'source': p['source']
        })

        dfs_to_merge.append(merged_df)
        print(f"Successfully merged {len(merged_df)} rows from {p['source']}.")

    except Exception as e:
        print(f"Error processing pair from source {p['source']}: {e}")
        exit(1)

if dfs_to_merge:
    final_dataset = pd.concat(dfs_to_merge, ignore_index=True)
    final_dataset.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"All datasets merged successfully. Total rows: {len(final_dataset)}")
    print(f"Merged dataset saved to: {OUTPUT_FILE}")
else :
    print("No datasets were merged. Please check for errors above.")

### 3.5. Enriquecimiento del dataset con URLs

In [ ]:
import pandas as pd
import random
import os
from google.colab import drive

DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'

SPAM_URLS_FILE1 = os.path.join(DRIVE_PATH, 'urls', 'SMS_Blacklist_OP18122025.txt')
SPAM_URLS_FILE2 = os.path.join(DRIVE_PATH, 'urls', 'SMS_Blacklist_OP31012026.txt')
HAM_URLS_FILE = os.path.join(DRIVE_PATH, 'urls', 'SMS_Whitelist_TrancoCleanReduced31012026.txt')

def load_lists():
    blacklist_urls = []
    blacklist_files = [SPAM_URLS_FILE1, SPAM_URLS_FILE2]
    for file in blacklist_files:
        if os.path.exists(file):
            with open(file, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                blacklist_urls.extend(lines)
        else:
            print(f"Warning: Blacklist file {file} does not exist.")
    
    whitelist_urls = []
    whitelist_files = [HAM_URLS_FILE]
    for file in whitelist_files:
        if os.path.exists(file):
            with open(file, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                whitelist_urls.extend(lines)
        else:
            print(f"Warning: Whitelist file {file} does not exist.")

    print (f"Loaded {len(whitelist_urls)} whitelist URLs and {len(blacklist_urls)} blacklist URLs.")

    return whitelist_urls, blacklist_urls

def get_connector(is_spam):
    if is_spam:
        connectors_en = ["Verify your account at", "Click the link:", "Visit", "Go to", "Check out", "Access your account here:", "Important notice:", "Action required:", "Limited time offer:", "Congratulations! You've won a prize at"]
        connectors_es = ["Verifica tu cuenta en", "Haz clic en el enlace:", "Visita", "Ve a", "Mira", "Accede a tu cuenta aquí:", "Aviso importante:", "Acción requerida:", "Oferta por tiempo limitado:", "¡Felicidades! Has ganado un premio en"]
    else:
        connectors_en = ["Check out", "Visit", "Go to", "Read more at", "Find out more at", "Learn more at"]
        connectors_es = ["Mira", "Visita", "Ve a", "Lee más en", "Descubre más en", "Aprende más en"]
    
    index = random.randint(0, len(connectors_en) - 1)
    
    selected_en = connectors_en[index]
    selected_es = connectors_es[index]

    return selected_en, selected_es

def enrich_sms(row, whitelist, blacklist):
    text_en = row['text_en']
    text_es = row['text_es']
    label = row['label']

    prob_url = 0.8 if label == 'spam' else 0.2

    if random.random() > prob_url:
        return text_en, text_es

    if label == 'spam':
        url = random.choice(blacklist)
    else:
        url = random.choice(whitelist)

    connector = get_connector(label == 'spam')

    return f"{text_en} {connector[0]} {url}", f"{text_es} {connector[1]} {url}"

def enrich_dataset(df, whitelist, blacklist):
    enriched_texts_en = []
    enriched_texts_es = []

    for _, row in df.iterrows():
        enriched_en, enriched_es = enrich_sms(row, whitelist, blacklist)
        enriched_texts_en.append(enriched_en)
        enriched_texts_es.append(enriched_es)

    df['text_en'] = enriched_texts_en
    df['text_es'] = enriched_texts_es

    return df

drive.mount('/content/drive')

white_list, black_list = load_lists()

if not white_list or not black_list:
    print("Error: Whitelist or Blacklist is empty. Cannot proceed with dataset enrichment.")
else:
    input_csv = os.path.join(DRIVE_PATH, 'datasets', 'bilingual_spam_sms_dataset.csv')
    if os.path.exists(input_csv):
        df_sms = pd.read_csv(input_csv)
        df_result = enrich_dataset(df_sms, white_list, black_list)

        output_csv = os.path.join(DRIVE_PATH, 'datasets', 'bilingual_spam_sms_dataset_enriched.csv')
        df_result.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"Enriched dataset saved to: {output_csv}")
    else:
        print(f"Error: Input dataset file {input_csv} does not exist.")